In [1]:
import pickle
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

c:\Users\vallu\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B")

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1880.49it/s]


In [3]:
dataset = pd.read_csv('../data/datasets/dataset_v1.csv')

In [4]:
dataset['context'] = dataset['context'].fillna('')
dataset['query'] = dataset['query'].fillna('')
dataset['label'] = dataset['label'].astype(int)

In [5]:
dataset.sample(5)

,query,context,label
72314,From the passage mention the different vertica...,"SoftBank Group Corp. (ソフトバンクグループ株式会社, SofutoBa...",1
44443,What is a bicameral legislature?,,1
8754,Under what dynasty was Caucasus established?,The first geographical entity that was called ...,0
44114,Suggest the best practice for minimizing the r...,,1
79530,What are some strategies for improving time ma...,,1


In [6]:
q_emb = embedding_model.encode(dataset['query'].tolist(), batch_size=32, show_progress_bar=True)
c_emb = embedding_model.encode(dataset['context'].tolist(), batch_size=8, show_progress_bar=True)

Batches: 100%|██████████| 10432/10432 [38:19<00:00,  4.54it/s] 


In [7]:
def cosine_similarity(vec1, vec2):
    vec1 = np.asarray(vec1)
    vec2 = np.asarray(vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

cos_sims = np.array([cosine_similarity(q, c) for q, c in zip(q_emb, c_emb)]).reshape(-1, 1)

In [8]:
diff = np.abs(q_emb - c_emb)
prod = q_emb * c_emb

X = np.hstack([diff, prod, cos_sims])
Y = dataset['label']

In [9]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)

In [10]:
model = XGBClassifier(
    max_depth=4,
    min_child_weight=10,
    gamma=1.0,
    subsample=0.8,
    colsample_bytree=0.5,
    n_estimators=1000,
    learning_rate=0.05,
    early_stopping_rounds=50,
    eval_metric='logloss'
)

In [11]:
model.fit(X_train, Y_train, eval_set=[(X_test, Y_test)], verbose=50)

[0]	validation_0-logloss:0.64633
[50]	validation_0-logloss:0.46917
[100]	validation_0-logloss:0.43136
[150]	validation_0-logloss:0.41271
[200]	validation_0-logloss:0.40132
[250]	validation_0-logloss:0.39332
[300]	validation_0-logloss:0.38824
[350]	validation_0-logloss:0.38379
[400]	validation_0-logloss:0.38046
[450]	validation_0-logloss:0.37749
[500]	validation_0-logloss:0.37507
[550]	validation_0-logloss:0.37311
[600]	validation_0-logloss:0.37148
[650]	validation_0-logloss:0.37016
[700]	validation_0-logloss:0.36879
[750]	validation_0-logloss:0.36755
[800]	validation_0-logloss:0.36647
[850]	validation_0-logloss:0.36536
[900]	validation_0-logloss:0.36449
[950]	validation_0-logloss:0.36393
[999]	validation_0-logloss:0.36320


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.5
,device,None
,early_stopping_rounds,50
,enable_categorical,True
,eval_metric,'logloss'


In [12]:
y_p = model.predict(X_train) 
acc = accuracy_score(Y_train, y_p)
print(f"Training Accuracy: {acc}")

Training Accuracy: 0.9067910850158768


In [13]:
y_p = model.predict(X_test) 
acc = accuracy_score(Y_test, y_p)
print(f"Validation Accuracy: {acc}")

Validation Accuracy: 0.8207417170930441


In [14]:
print(classification_report(Y_test, y_p))

              precision    recall  f1-score   support

           0       0.75      0.76      0.76      6149
           1       0.86      0.85      0.86     10542

    accuracy                           0.82     16691
   macro avg       0.81      0.81      0.81     16691
weighted avg       0.82      0.82      0.82     16691



In [15]:
emd_dim = q_emb[0].shape[0]

feature_names = (
    [f"diff_{i+1}" for i in range(emd_dim)] +
    [f"mul_{i+1}" for i in range(emd_dim)] +
    ["cos_sims"]
)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance_df.head(20))

        feature  importance
7        diff_8    0.014796
1059     mul_36    0.012084
1394    mul_371    0.010724
1321    mul_298    0.007458
1026      mul_3    0.006904
429    diff_430    0.005915
1467    mul_444    0.005815
1360    mul_337    0.005248
1129    mul_106    0.004799
1037     mul_14    0.004673
1453    mul_430    0.004232
1889    mul_866    0.003884
15      diff_16    0.003858
1027      mul_4    0.003807
1086     mul_63    0.003619
14      diff_15    0.003557
1109     mul_86    0.003439
2048   cos_sims    0.003179
13      diff_14    0.002969
1004  diff_1005    0.002883


In [17]:
real_examples = [
    ("what was the YOLO config we used", "we used YOLOv11n with conf=0.6 freeze=15 imgsz=320"),
    ("what was the learning rate", "we set learning rate to 0.001 with cosine annealing scheduler"),
    ("what was the YOLO config we used", "the capital of France is Paris"),
    ("what was the learning rate", "Beyonce was born in Houston Texas in 1981"),
    ("what was the YOLO config we used", ""),
    ("continue from where we left off", ""),
    ("what did we decide about the database", ""),
    ("what is 2+2", ""),
    ("who invented python", ""),
    ("what is the capital of France", ""),
    ("what is 2+2", "Beyonce was born in Houston Texas"),
    ("who invented python", "the capital of France is Paris"),
]

print(f"\n{'Query':<45} {'Context':<40} {'Predicted'}")
print("-" * 95)

for query, context in real_examples:
    q = embedding_model.encode(query, convert_to_numpy=True)
    c = embedding_model.encode(context, convert_to_numpy=True)

    # cosine similarity (manual, safe)
    norm_q = np.linalg.norm(q)
    norm_c = np.linalg.norm(c)

    if norm_q == 0 or norm_c == 0:
        cos_sim = 0.0
    else:
        cos_sim = np.dot(q, c) / (norm_q * norm_c)

    X_pred = np.hstack([np.abs(q - c), c*q, [cos_sim]])

    pred = model.predict(X_pred.reshape(1, -1))[0]

    print(f"{query[:44]:<45} {context[:39]:<40} {pred}")


Query                                         Context                                  Predicted
-----------------------------------------------------------------------------------------------
what was the YOLO config we used              we used YOLOv11n with conf=0.6 freeze=1  1
what was the learning rate                    we set learning rate to 0.001 with cosi  1
what was the YOLO config we used              the capital of France is Paris           0
what was the learning rate                    Beyonce was born in Houston Texas in 19  0
what was the YOLO config we used                                                       0
continue from where we left off                                                        0
what did we decide about the database                                                  0
what is 2+2                                                                            1
who invented python                                                                    1
what 

In [18]:
with open("../models/XGBoost_82.pkl", "wb") as f:
    pickle.dump(model, f)